In [2]:
"""
Build the post-level dataset for testing whether the burnout-severity to
exit-intention relationship is MODERATED by crisis intensity (a rolling
count of severe CVEs from the full NVD database, not just the 4 named
anchor events).

Why a rolling count instead of "days since last severe CVE": Run1 (>9.5)
has ~6 CVEs/day on average -- there's almost never a gap, so "days since
last one" would be meaningless (always ~0). A rolling N-day COUNT still
varies meaningfully (some 2-week stretches have far more severe CVEs than
others), and works for both the dense Run1 threshold and the rarer Run2
(==10) threshold.

Output: crisis_intensity_post_level.csv, one row per burnout post:
  post_id, date, bat_score, has_exit_intent (0/1),
  crisis_intensity_run1, crisis_intensity_run2 (rolling 14-day CVE counts
  ending on that post's date), days_since_start (linear time control)
"""

import pandas as pd
import numpy as np
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
BAT_PATH = "/Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates.csv"
EXIT_PATH = "/Users/nadia/Desktop/redditRun_june/exit_annotated_pass1.csv"
NVD_PATH = "/Users/nadia/Desktop/redditRun_june/nvd_chunks/nvd_merged_full.csv"
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q3_analysis/"

ROLLING_WINDOW_DAYS = 14  # matches the "crisis" window length used elsewhere


def load_bat():
    cols = ["post_id", "EX", "EMO", "COG", "MD", "bat_score", "created_date"]
    df = pd.read_csv(BAT_PATH, usecols=cols)
    return df.rename(columns={"created_date": "created_date_bat"})


def load_exit():
    cols = ["id", "exit_intention_class", "created_date"]
    df = pd.read_csv(EXIT_PATH, usecols=cols)
    return df.rename(columns={"id": "post_id", "created_date": "created_date_exit"})


def build_daily_cve_counts(nvd_path):
    df = pd.read_csv(nvd_path, usecols=["cve_id", "published", "cvss_base_score"])
    df["date"] = pd.to_datetime(df["published"], errors="coerce").dt.date
    df["is_run1"] = (df["cvss_base_score"] > 9.5).astype(int)
    df["is_run2"] = (df["cvss_base_score"] >= 10).astype(int)
    daily = df.groupby("date").agg(
        n_cve_run1=("is_run1", "sum"),
        n_cve_run2=("is_run2", "sum"),
    ).reset_index()
    return daily


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    print("Loading + merging BAT and exit intention...")
    bat = load_bat()
    exit_df = load_exit()
    merged = bat.merge(exit_df, on="post_id", how="inner", validate="1:1")
    print(f"  Merged rows: {len(merged)} (bat: {len(bat)}, exit: {len(exit_df)})")

    merged["date"] = pd.to_datetime(merged["created_date_bat"], errors="coerce").dt.date
    merged["has_exit_intent"] = merged["exit_intention_class"].isin(
        ["exit_explicit", "exit_contemplating"]
    ).astype(int)

    print("\nBuilding daily CVE counts + rolling crisis intensity...")
    daily_cve = build_daily_cve_counts(NVD_PATH)

    # Build a continuous daily calendar covering the full range, zero-filled,
    # so the rolling window has no gaps to silently skip over.
    start = min(merged["date"].min(), daily_cve["date"].min())
    end = max(merged["date"].max(), daily_cve["date"].max())
    full_index = pd.DataFrame({"date": pd.date_range(start, end, freq="D").date})
    daily_cve_full = full_index.merge(daily_cve, on="date", how="left")
    daily_cve_full["n_cve_run1"] = daily_cve_full["n_cve_run1"].fillna(0)
    daily_cve_full["n_cve_run2"] = daily_cve_full["n_cve_run2"].fillna(0)

    daily_cve_full = daily_cve_full.sort_values("date").reset_index(drop=True)
    daily_cve_full["crisis_intensity_run1"] = (
        daily_cve_full["n_cve_run1"].rolling(window=ROLLING_WINDOW_DAYS, min_periods=1).sum()
    )
    daily_cve_full["crisis_intensity_run2"] = (
        daily_cve_full["n_cve_run2"].rolling(window=ROLLING_WINDOW_DAYS, min_periods=1).sum()
    )

    print(f"  crisis_intensity_run1: min={daily_cve_full['crisis_intensity_run1'].min():.0f}, "
          f"max={daily_cve_full['crisis_intensity_run1'].max():.0f}, "
          f"mean={daily_cve_full['crisis_intensity_run1'].mean():.1f}")
    print(f"  crisis_intensity_run2: min={daily_cve_full['crisis_intensity_run2'].min():.0f}, "
          f"max={daily_cve_full['crisis_intensity_run2'].max():.0f}, "
          f"mean={daily_cve_full['crisis_intensity_run2'].mean():.1f}")

    print("\nJoining crisis intensity onto each post by date...")
    result = merged.merge(
        daily_cve_full[["date", "crisis_intensity_run1", "crisis_intensity_run2"]],
        on="date", how="left"
    )
    n_unmatched = result["crisis_intensity_run1"].isna().sum()
    if n_unmatched:
        print(f"  WARNING: {n_unmatched} posts couldn't be matched to a crisis intensity value")

    result["days_since_start"] = (pd.to_datetime(result["date"]) - pd.to_datetime(start)).dt.days

    keep_cols = ["post_id", "date", "bat_score", "EX", "EMO", "COG", "MD",
                 "exit_intention_class", "has_exit_intent",
                 "crisis_intensity_run1", "crisis_intensity_run2", "days_since_start"]
    result = result[keep_cols]

    out_path = os.path.join(OUT_DIR, "crisis_intensity_post_level.csv")
    result.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    print(f"Final post-level table: {len(result)} rows")
    print("\nhas_exit_intent distribution:")
    print(result["has_exit_intent"].value_counts().to_string())


if __name__ == "__main__":
    main()

Loading + merging BAT and exit intention...
  Merged rows: 135897 (bat: 135897, exit: 135897)

Building daily CVE counts + rolling crisis intensity...
  crisis_intensity_run1: min=3, max=278, mean=79.4
  crisis_intensity_run2: min=0, max=17, mean=2.9

Joining crisis intensity onto each post by date...

Saved -> /Users/nadia/Desktop/redditRun_june/q3_analysis/crisis_intensity_post_level.csv
Final post-level table: 135897 rows

has_exit_intent distribution:
has_exit_intent
0    125679
1     10218
